# 10. Final project: an end-to-end image-analysis workflow

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner

## What you will learn
- Define an analysis question
- Inspect and preprocess intentionally
- Segment and validate before measurement
- Test parameter sensitivity
- Save results and analysis settings

> **Learning rule:** understand the problem first, then choose the function.

## 1. Define the question

**Question:** Can we identify bright cell-like regions in a grayscale teaching image and measure their area and intensity?

A clear question defines what counts as foreground and what measurements matter.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import skimage as ski
from skimage import filters, morphology, measure, color
image = ski.data.cell()
print("shape:", image.shape, "dtype:", image.dtype, "range:", (image.min(), image.max()))
plt.imshow(image,cmap="gray"); plt.title("Raw image"); plt.axis("off"); plt.show()

## 2. Inspect → preprocess → segment

Each step below has one purpose. Do not add operations simply because they are available.

In [ ]:
# Gaussian sigma=1.2 is a modest smoothing choice; inspect whether small structures survive.
smooth = filters.gaussian(image, sigma=1.2)
# Otsu is a reasonable first global threshold when bright/dark classes may separate.
threshold = filters.threshold_otsu(smooth)
mask = smooth > threshold
# Clean only very small components/holes; these parameters are in pixels.
mask = morphology.remove_small_objects(mask, max_size=39)
mask = morphology.remove_small_holes(mask, max_size=39)
labels = measure.label(mask)

## 3. Validate → measure

Overlay the labels on the source before trusting the measurement table.

In [ ]:
# Overlay is the critical QC step before measurement.
overlay = color.label2rgb(labels, image=image, bg_label=0, alpha=0.35)
plt.imshow(overlay); plt.title(f"QC overlay: {labels.max()} labels"); plt.axis("off"); plt.show()
props = measure.regionprops_table(labels, intensity_image=image, properties=("label","area","centroid","mean_intensity","eccentricity","solidity"))
results = pd.DataFrame(props)
print(results.head())

## 4. Sensitivity and reproducibility

Small reasonable parameter changes should not cause unexplained catastrophic changes. Save both results and settings.

In [ ]:
rows=[]
for sigma in [0.5,1.0,1.5,2.0]:
    test = filters.gaussian(image, sigma=sigma)
    t = filters.threshold_otsu(test)
    m = morphology.remove_small_objects(test > t, max_size=39)
    lab = measure.label(m)
    rows.append({"sigma":sigma,"threshold":float(t),"object_count":int(lab.max()),"foreground_fraction":float(m.mean())})
sensitivity = pd.DataFrame(rows)
print(sensitivity)

out=Path("../outputs"); out.mkdir(exist_ok=True)
results.to_csv(out/"final_project_measurements.csv", index=False)
params={"source":"skimage.data.cell","gaussian_sigma":1.2,"threshold":"Otsu","threshold_value":float(threshold),"cleanup_max_size":39,"scikit_image_version":ski.__version__}
(out/"final_project_parameters.json").write_text(json.dumps(params,indent=2))
print("saved results and parameters")

## Function-selection guide

| Decision | Tool | Validation question |
|---|---|---|
| Understand input | `shape`, `dtype`, histogram | Do I understand the data? |
| Reduce fine noise | `filters.gaussian()` | Did I erase real structure? |
| Bright/dark separation | `threshold_otsu()` | Does one global split fit? |
| Remove tiny artifacts | morphology cleanup | Am I deleting real objects? |
| Identify objects | `measure.label()` | Are merges/splits acceptable? |
| Quantify | `regionprops_table()` | Are labels scientifically valid? |
| Reproducibility | save CSV + JSON + versions | Could another person repeat it? |

## Takeaway

**Choose functions because they solve a specific image problem, and always inspect the result before measuring.**